# Assignment -- Cedar Grove Public Library: Checkouts

## The scenario

Cedar Grove Public Library tracks every book checkout in `data/checkouts.csv`: who
checked out which book, when it was due back, when (if ever) it was returned, and any
late fee charged. The head librarian has five questions.

In [2]:
import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_columns", 20)

---
## Problem 1 (basic) -- Load and get oriented

In [3]:
checkouts_df = pd.read_csv(
    "data/checkouts.csv",
    parse_dates=["checkout_date", "due_date", "return_date"],
)

n_total_checkouts = len(checkouts_df)
n_still_checked_out = checkouts_df["return_date"].isna().sum()

print(f"{n_total_checkouts} total checkouts, {n_still_checked_out} still checked out")

160 total checkouts, 48 still checked out


In [4]:
# Check yourself
assert n_total_checkouts == len(checkouts_df)
assert n_still_checked_out == checkouts_df["return_date"].isna().sum()
assert n_still_checked_out < n_total_checkouts
print("Looks good.")

Looks good.


---
## Problem 2 (basic-medium) -- Clean the data, the right way for each column

In [5]:
checkouts_clean = checkouts_df.copy()

checkouts_clean["is_returned"] = checkouts_clean["return_date"].notna()

checkouts_clean["late_fee"] = checkouts_clean["late_fee"].fillna(0)

checkouts_clean.head()

,checkout_id,member_id,book_title,genre,checkout_date,due_date,return_date,late_fee,is_returned
0,CHK-3001,MEM-0014,The Great Gatsby,Classic Fiction,2026-06-12,2026-07-03,2026-07-03,0.0,True
1,CHK-3002,MEM-0034,Jane Eyre,Gothic,2026-07-19,2026-08-09,NaT,0.0,False
2,CHK-3003,MEM-0060,The Catcher in the Rye,Classic Fiction,2026-01-02,2026-01-23,NaT,0.0,False
3,CHK-3004,MEM-0051,The Hobbit,Adventure,2026-06-16,2026-07-07,NaT,0.0,False
4,CHK-3005,MEM-0028,The Catcher in the Rye,Classic Fiction,2026-04-01,2026-04-22,2026-04-22,0.0,True


In [6]:
# Check yourself
assert checkouts_clean["late_fee"].isna().sum() == 0
assert checkouts_clean["is_returned"].dtype == bool
assert checkouts_clean["is_returned"].sum() == checkouts_clean["return_date"].notna().sum()
print("Looks good:", checkouts_clean["is_returned"].value_counts().to_dict())

Looks good: {True: 112, False: 48}


---
## Problem 3 (medium) -- Which genre racks up the most late fees?

In [7]:
returned_only = checkouts_clean[checkouts_clean["is_returned"]]
avg_late_fee_by_genre = (
    returned_only.groupby("genre")["late_fee"].mean().sort_values(ascending=False)
)
avg_late_fee_by_genre

genre
Dystopian             0.670455
Historical Fiction    0.625000
Adventure             0.565789
Gothic                0.433333
Classic Fiction       0.291667
Name: late_fee, dtype: float64

In [8]:
# Check yourself
assert len(avg_late_fee_by_genre) == checkouts_clean["genre"].nunique()
assert avg_late_fee_by_genre.is_monotonic_decreasing
print("Looks good -- worst genre for late fees:", avg_late_fee_by_genre.idxmax())

Looks good -- worst genre for late fees: Dystopian


---
## Problem 4 (basic-medium) -- Look up each book with a real public API

In [9]:
BACKUP_BOOK_FACTS = {
    "Pride and Prejudice": {"author": "Jane Austen", "first_publish_year": 1813},
    "To Kill a Mockingbird": {"author": "Harper Lee", "first_publish_year": 1960},
    "The Great Gatsby": {"author": "F. Scott Fitzgerald", "first_publish_year": 1925},
    "The Catcher in the Rye": {"author": "J. D. Salinger", "first_publish_year": 1951},
    "1984": {"author": "George Orwell", "first_publish_year": 1949},
    "Brave New World": {"author": "Aldous Huxley", "first_publish_year": 1932},
    "Frankenstein": {"author": "Mary Shelley", "first_publish_year": 1818},
    "Jane Eyre": {"author": "Charlotte Bronte", "first_publish_year": 1847},
    "Moby Dick": {"author": "Herman Melville", "first_publish_year": 1851},
    "The Hobbit": {"author": "J. R. R. Tolkien", "first_publish_year": 1937},
    "War and Peace": {"author": "Leo Tolstoy", "first_publish_year": 1869},
    "Crime and Punishment": {"author": "Fyodor Dostoevsky", "first_publish_year": 1866},
}

OPEN_LIBRARY_API = "https://openlibrary.org/search.json"

In [10]:
def get_book_facts(title):
    """Return {"author", "first_publish_year"} for the top Open Library match.

    Falls back to BACKUP_BOOK_FACTS[title] if the request fails or the response
    doesn't have the fields we expect.
    """
    try:
        response = requests.get(OPEN_LIBRARY_API, params={"q": title}, timeout=10)
        response.raise_for_status()

        top_match = response.json()["docs"][0]

        return {
            "author": top_match["author_name"][0],
            "first_publish_year": int(top_match["first_publish_year"]),
        }

    except (requests.RequestException, ValueError, KeyError, IndexError, TypeError):
        return BACKUP_BOOK_FACTS[title]

get_book_facts("1984")

{'author': 'George Orwell', 'first_publish_year': 1949}

Now call it for every distinct title in the library's catalog and assemble `book_facts_df`,
indexed by `book_title`, with columns `author` and `first_publish_year`.

In [11]:
records = {}
for title in checkouts_clean["book_title"].unique():
    records[title] = get_book_facts(title)

book_facts_df = pd.DataFrame(records).T
book_facts_df.index.name = "book_title"
book_facts_df

,author,first_publish_year
book_title,,
The Great Gatsby,F. Scott Fitzgerald,1920
Jane Eyre,Charlotte Brontë,1847
The Catcher in the Rye,J. D. Salinger,1945
The Hobbit,J.R.R. Tolkien,1937
Crime and Punishment,Фёдор Достоевский,1866
War and Peace,Лев Толстой,1864
To Kill a Mockingbird,Harper Lee,1960
Moby Dick,Herman Melville,1851
Brave New World,Aldous Huxley,1932


In [12]:
# Check yourself
assert len(book_facts_df) == checkouts_clean["book_title"].nunique()
assert set(["author", "first_publish_year"]).issubset(book_facts_df.columns)
print("Looks good:", book_facts_df.shape)

Looks good: (12, 2)


---
## Problem 5 (medium) -- Which author costs the library the most in late fees?

In [13]:
checkouts_with_author = checkouts_clean.merge(
    book_facts_df.reset_index(),
    on="book_title",
    how="left",
)

late_fee_by_author = (
    checkouts_with_author.groupby("author")["late_fee"].sum().sort_values(ascending=False)
)
late_fee_by_author

author
Aldous Huxley          7.75
George Orwell          7.00
Фёдор Достоевский      6.50
Herman Melville        6.50
Лев Толстой            6.00
J. D. Salinger         5.75
J.R.R. Tolkien         4.25
Charlotte Brontë      4.25
F. Scott Fitzgerald    2.25
Mary Shelley           2.25
Harper Lee             1.25
Jane Austen            1.25
Name: late_fee, dtype: float64

In [14]:
# Check yourself
assert len(checkouts_with_author) == len(checkouts_clean)
assert late_fee_by_author.is_monotonic_decreasing
print("Looks good -- costliest author:", late_fee_by_author.idxmax())

Looks good -- costliest author: Aldous Huxley
